# Wall C annotated hold evaluation

Compare a saved **3D** DeMoSearch run with the annotated holds in Google Drive. Use a CPU runtime and keep the predicted `scene.json` and its companion GLB together. This notebook works with existing reconstruction and inference results.

| Route | Ground truth in Wall C |
| --- | --- |
| Blue V5 | 9 named GLB holds and a JSON with geometry and start/finish labels |
| Yellow V3 | 13 named GLB holds. The file also includes the same 9 Blue V5 nodes |
| Green V1 | Reference photo. A marked GLB is needed for 3D scoring |

The annotation files label individual routes. Treat results as candidate geometric matches. Review which annotated holds are in view before using a recall denominator.

In [ ]:
import importlib.util
import json
import subprocess
import sys
from pathlib import Path

ON_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if ON_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

packages = ("numpy", "scipy", "trimesh", "pandas", "plotly")
missing = [package for package in packages if importlib.util.find_spec(package) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import trimesh
from IPython.display import display
from scipy.optimize import linear_sum_assignment

## 1. Choose a capture

The default run path matches the Wall C 3D preset in the inference notebook. A separate Colab runtime has its own `/content` directory. In that case, save both files from your inference run together in Drive and set `PREDICTED_SCENE` to that copy. If you uploaded them to the current session, use the uploaded paths.

Set `VISIBLE_HOLDS` after checking the frames from the prediction run. Enter a list of visible annotation IDs for each route, or use `"all"` after confirming the whole route is in view. The notebook assigns confirmed visible holds first and uses remaining predictions to explore routes with unconfirmed visibility. For a crop of Blue V5, set `ROUTES = ("blue_v5",)` to score Blue alone.

In [ ]:
WALL_C = Path("/content/drive/MyDrive/CV26_FinalProj/DeMoSearch-data/wallC_overhang_right")
RUN_DIR = Path("/content/demosearch-inference/results/wallc_section_3d")
PREDICTED_SCENE = RUN_DIR / "scene.json"
PREDICTED_GLB = None  # Optional path to the exact GLB exported with the scene.
ROUTES = ("blue_v5", "yellow_v3", "green_v1")
VISIBLE_HOLDS = {}  # Example {"blue_v5": ["blue_v5_holds.001", "blue_v5_holds.005"]}
MAX_DISTANCE_M = 0.10
OUTPUT_DIR = RUN_DIR / "annotated_evaluation"

In [ ]:
def annotated_centers(scene):
    centers = {}
    for node in sorted(scene.graph.nodes_geometry):
        pose, name = scene.graph[node]
        mesh = scene.geometry[name]
        identifier = mesh.metadata.get("hold_id") or node
        if "_holds." not in identifier.casefold() or not len(mesh.vertices):
            continue
        if identifier in centers:
            raise ValueError(f"Duplicate hold ID {identifier}")
        points = trimesh.transform_points(mesh.vertices, pose)
        centers[identifier] = np.median(points, axis=0)
    return centers


def load_references(root, routes, visible_holds):
    if not root.is_dir():
        raise FileNotFoundError(f"Set WALL_C to the mounted Wall C folder. Current path {root}")
    if len(set(routes)) != len(routes) or set(visible_holds) - set(routes):
        raise ValueError("Choose unique ROUTES and specify visibility only for selected routes")
    folders = {path.name.casefold(): path for path in root.iterdir() if path.is_dir()}
    assets, shared, target_rows = {}, {}, []
    for route in routes:
        folder = folders.get(route.casefold())
        files = sorted(folder.iterdir()) if folder else []
        glbs = [path for path in files if path.is_file() and path.name.casefold().endswith("_marked.glb")]
        photos = [path for path in files if path.suffix.lower() in {".heic", ".jpg", ".jpeg", ".png"}
                  and path.stem.casefold() == route.casefold()]
        if len(glbs) > 1:
            raise ValueError(f"{route} has multiple marked GLBs. Choose one reference file.")
        info = {
            "route": route, "photo": str(photos[0]) if photos else None,
            "glb": str(glbs[0]) if glbs else None, "annotation_count": 0,
            "reference_status": "missing_marked_glb", "scope_confirmed": False,
        }
        assets[route] = info
        if not glbs:
            continue

        centers = annotated_centers(trimesh.load_scene(glbs[0], process=False))
        if shared:
            overlap = shared.keys() & centers.keys()
            if not overlap or max(np.linalg.norm(shared[key] - centers[key]) for key in overlap) > 0.005:
                raise ValueError("The selected GLBs have no verified shared frame. Evaluate one route at a time.")
        shared.update(centers)
        route_centers = {key: center for key, center in centers.items()
                         if key.casefold().startswith(f"{route.casefold()}_holds.")}
        scope = visible_holds.get(route)
        if scope is not None and scope != "all":
            if not isinstance(scope, list) or set(scope) - route_centers.keys():
                raise ValueError(f"VISIBLE_HOLDS for {route} must be 'all' or a list of its annotation IDs")
        info.update(
            annotation_count=len(route_centers), scope_confirmed=scope is not None,
            reference_status="annotated" if route_centers else "no_route_hold_nodes",
        )
        for key, center in route_centers.items():
            target_rows.append({
                "route": route, "annotated_hold_id": key, "target_position_m": center.tolist(),
                "visibility": "unknown" if scope is None else "visible" if scope == "all" or key in scope else "out_of_scope",
            })
    return assets, target_rows


def load_blue_metadata(root, rows):
    path = root / "blue_v5" / "wallC_bluev5_holds.json"
    if not path.is_file():
        return None
    document = json.loads(path.read_text())
    blue_targets = {row["annotated_hold_id"]: row for row in rows if row["route"] == "blue_v5"}
    for hold in document["holds"]:
        if hold["id"] in blue_targets:
            row = blue_targets[hold["id"]]
            row["reference"] = {
                "height_m": hold.get("height_m"), "protrusion_m": hold.get("protrusion_m"),
                "is_start": hold["id"] in document.get("start_hold_ids", []),
                "is_top": hold["id"] == document.get("top_hold_id"),
            }
    return str(path)


assets, targets = load_references(WALL_C, ROUTES, VISIBLE_HOLDS)
blue_metadata = load_blue_metadata(WALL_C, targets)
display(pd.DataFrame(assets.values())[["route", "reference_status", "annotation_count", "scope_confirmed"]])

scene = json.loads(PREDICTED_SCENE.read_text()) if PREDICTED_SCENE.is_file() else None
prediction_glb = Path(PREDICTED_GLB) if PREDICTED_GLB else None
if scene is None:
    print(f"Set PREDICTED_SCENE to a saved 3D scene. Current path {PREDICTED_SCENE}")
elif scene["analysis_mode"] != "3d":
    raise ValueError("Select the scene.json from a 3D run")
elif original := scene.get("wall", {}).get("glb_path"):
    original = Path(original)
    if prediction_glb is not None and prediction_glb.name != original.name:
        raise ValueError(f"Use the scene's companion {original.name} for the prediction GLB")
    if prediction_glb is None:
        beside_scene = PREDICTED_SCENE.parent / original.name
        prediction_glb = beside_scene if beside_scene.is_file() else original

## 2. Align the two walls

Each reconstruction has its own position, rotation and scale. Enter an existing transform in `ALIGNMENT_JSON` or use the two plots to build one. In each plot, hover over the same recognizable point on the wall to read its XYZ values. Copy at least three non-collinear pairs into `FIT_LANDMARK_PAIRS`. Use features spread across the shared area, such as panel corners or texture marks away from the annotated holds. Optional `CHECK_LANDMARK_PAIRS` let you measure alignment error at independent locations.

The transform takes coordinates from the **prediction GLB**, the same frame as `scene.json`'s `position_glb`, to the annotated wall in meters. Enter each pair as `([prediction_x, prediction_y, prediction_z], [annotation_x, annotation_y, annotation_z])`. Keep each file in its original coordinates when choosing points.

In [ ]:
def wall_trace(path, label, max_points=12000):
    mesh = trimesh.load_scene(path, process=False).to_mesh()
    vertices = np.asarray(mesh.vertices)
    if not len(vertices):
        raise ValueError(f"No vertices in {path}")
    rng = np.random.default_rng(42)
    indices = rng.choice(len(vertices), min(max_points, len(vertices)), replace=False)
    points = vertices[indices]
    if mesh.visual.kind == "texture":
        colors = mesh.visual.to_color().vertex_colors[indices, :3]
    elif mesh.visual.kind == "vertex":
        colors = mesh.visual.vertex_colors[indices, :3]
    else:
        colors = None
    marker_color = [f"rgb({r},{g},{b})" for r, g, b in colors] if colors is not None else "#64748b"
    return go.Scatter3d(
        x=points[:, 0], y=points[:, 1], z=points[:, 2], mode="markers", name=label,
        marker={"size": 2, "color": marker_color},
        hovertemplate="x %{x:.5f}<br>y %{y:.5f}<br>z %{z:.5f}<extra></extra>",
    )


def configure_plot(figure, title, front):
    figure.update_layout(
        title=title, height=580, margin={"l": 0, "r": 0, "b": 0, "t": 40},
        scene={"aspectmode": "data", "camera": {"up": {"x": 0, "y": 1, "z": 0},
                                                "eye": {"x": 0, "y": 0, "z": front}}},
    )


reference_glb = next((Path(info["glb"]) for info in assets.values() if info["glb"]), None)
annotation_wall = wall_trace(reference_glb, "Annotated wall") if reference_glb else None
if prediction_glb and prediction_glb.is_file():
    figure = go.Figure(wall_trace(prediction_glb, "Prediction wall"))
    configure_plot(figure, "Prediction GLB", front=2.5)
    figure.show()
elif scene is not None:
    print("Set PREDICTED_GLB to the matching reconstruction to pick landmarks.")
if annotation_wall is not None:
    figure = go.Figure(annotation_wall)
    configure_plot(figure, "Annotation GLB", front=-2.5)
    figure.show()

In [ ]:
ALIGNMENT_JSON = None  # Optional JSON containing a 4 x 4 prediction-GLB-to-annotation matrix.
FIT_LANDMARK_PAIRS = []
CHECK_LANDMARK_PAIRS = []


def landmark_arrays(pairs):
    values = np.asarray(pairs, dtype=float)
    if values.ndim != 3 or values.shape[1:] != (2, 3) or not np.isfinite(values).all():
        raise ValueError("Each landmark pair must give two finite XYZ points")
    return values[:, 0], values[:, 1]


def fit_similarity(pairs):
    source, target = landmark_arrays(pairs)
    a, b = source - source.mean(axis=0), target - target.mean(axis=0)
    if len(source) < 3 or min(np.linalg.matrix_rank(a), np.linalg.matrix_rank(b)) < 2:
        raise ValueError("Choose at least three well-spread, non-collinear landmark pairs")
    u, singular, vt = np.linalg.svd(a.T @ b)
    signs = np.ones(3)
    signs[-1] = np.linalg.det(vt.T @ u.T)
    rotation = vt.T @ np.diag(signs) @ u.T
    scale = float(singular @ signs / np.sum(a ** 2))
    matrix = np.eye(4)
    matrix[:3, :3] = scale * rotation
    matrix[:3, 3] = target.mean(axis=0) - matrix[:3, :3] @ source.mean(axis=0)
    return matrix


def validate_matrix(matrix):
    matrix = np.asarray(matrix, dtype=float)
    if (matrix.shape != (4, 4) or not np.isfinite(matrix).all()
            or not np.allclose(matrix[3], [0, 0, 0, 1])):
        raise ValueError("Enter a finite 4 x 4 homogeneous transform")
    linear = matrix[:3, :3]
    singular = np.linalg.svd(linear, compute_uv=False)
    if singular[-1] <= 0 or np.linalg.det(linear) <= 0 or singular[0] / singular[-1] > 1.001:
        raise ValueError("The transform must have one positive scale and a proper rotation")
    return matrix


def alignment_errors(pairs, matrix):
    if not pairs:
        return []
    source, target = landmark_arrays(pairs)
    return np.linalg.norm(trimesh.transform_points(source, matrix) - target, axis=1).tolist()


alignment = None
if ALIGNMENT_JSON is not None:
    document = json.loads(Path(ALIGNMENT_JSON).read_text())
    if isinstance(document, dict) and document.get("from", "prediction_position_glb") != "prediction_position_glb":
        raise ValueError("The alignment must start in the prediction GLB frame")
    matrix = validate_matrix(document["matrix"] if isinstance(document, dict) else document)
    source = "provided_matrix"
elif FIT_LANDMARK_PAIRS:
    matrix = validate_matrix(fit_similarity(FIT_LANDMARK_PAIRS))
    source = "landmark_similarity"
else:
    matrix = None
    print("Enter landmark pairs or supply ALIGNMENT_JSON before evaluating distances.")

if matrix is not None:
    alignment = {
        "matrix": matrix.tolist(), "source": source, "from": "prediction_position_glb",
        "to": "annotated_glb_meters", "fit_errors_m": alignment_errors(FIT_LANDMARK_PAIRS, matrix),
        "check_errors_m": alignment_errors(CHECK_LANDMARK_PAIRS, matrix),
    }
    print("Fit errors in cm", np.round(np.array(alignment["fit_errors_m"]) * 100, 1))
    if alignment["check_errors_m"]:
        print("Independent check errors in cm", np.round(np.array(alignment["check_errors_m"]) * 100, 1))
    else:
        print("Independent alignment error is unknown. Treat the scores as provisional.")

## 3. Run the comparison

Each prediction and each annotated hold can appear in at most one match. The distance threshold is in annotation meters. Marker positions are centers of the predicted and annotated holds. Inspect the lines and unmatched visible holds to validate the matchup. Unassigned predictions may belong to other routes, so the report lists their count. Recall requires confirmed visibility and independent alignment checks within the match threshold.

The Blue JSON's reported heights, protrusions and route tags are attached to its rows for review. Distances use GLB mesh centers for both routes so each route is scored the same way.

In [ ]:
def compare_holds(scene, targets, alignment, maximum_distance_m):
    if not np.isfinite(maximum_distance_m) or maximum_distance_m <= 0:
        raise ValueError("MAX_DISTANCE_M must be positive and finite")
    located = [hold for hold in scene["holds"] if hold.get("position_glb") is not None]
    if scene["holds"] and not located:
        raise ValueError("This scene has no GLB positions. Select an exported 3D run with mapped holds.")
    positions = np.empty((0, 3))
    if located:
        positions = np.asarray([hold["position_glb"] for hold in located], dtype=float)
        if positions.shape != (len(located), 3) or not np.isfinite(positions).all():
            raise ValueError("Each mapped hold needs one finite GLB XYZ position")
        positions = trimesh.transform_points(positions, np.asarray(alignment["matrix"]))

    default_status = {"visible": "unmatched", "unknown": "visibility_unknown", "out_of_scope": "out_of_scope"}
    rows = [dict(target, prediction_id=None, prediction_position_m=None, distance_m=None,
                 status=default_status[target["visibility"]]) for target in targets]
    remaining = list(range(len(positions)))
    for visibility, status in [("visible", "matched"), ("unknown", "tentative_match")]:
        eligible = [row for row in rows if row["visibility"] == visibility]
        if not eligible or not remaining:
            continue
        expected = np.asarray([row["target_position_m"] for row in eligible])
        distances = np.linalg.norm(positions[remaining, None] - expected[None, :], axis=-1)
        # Make every valid assignment cheaper than giving up one valid match.
        invalid_cost = (min(distances.shape) + 1) * maximum_distance_m
        cost = np.where(distances <= maximum_distance_m, distances, invalid_cost)
        assigned = set()
        for i, j in zip(*linear_sum_assignment(cost)):
            if distances[i, j] <= maximum_distance_m:
                prediction = remaining[i]
                eligible[j].update(
                    prediction_id=located[prediction]["id"],
                    prediction_position_m=positions[prediction].tolist(),
                    distance_m=float(distances[i, j]), status=status,
                )
                assigned.add(prediction)
        remaining = [index for index in remaining if index not in assigned]
    return rows, positions, len(scene["holds"]) - len(located)


def summarize_routes(assets, rows, report_recall):
    summary = []
    for route, info in assets.items():
        eligible = [row for row in rows if row["route"] == route and row["visibility"] == "visible"]
        errors = [row["distance_m"] for row in eligible if row["distance_m"] is not None]
        confirmed = info["scope_confirmed"] and info["reference_status"] == "annotated"
        tentative = sum(row["route"] == route and row["status"] == "tentative_match" for row in rows)
        summary.append({
            "route": route, "reference_status": info["reference_status"],
            "annotated": info["annotation_count"], "confirmed_visible": len(eligible) if confirmed else None,
            "matched": len(errors) if confirmed else None,
            "unmatched": len(eligible) - len(errors) if confirmed else None,
            "tentative_matches": tentative if info["reference_status"] == "annotated" and not confirmed else None,
            "geometric_recall": len(errors) / len(eligible) if eligible and confirmed and report_recall else None,
            "median_error_cm": float(np.median(errors) * 100) if errors else None,
            "p90_error_cm": float(np.percentile(errors, 90) * 100) if errors else None,
        })
    return summary


report = None
if scene is not None and alignment is not None and targets:
    rows, aligned_positions, unmapped = compare_holds(scene, targets, alignment, MAX_DISTANCE_M)
    check_errors = alignment.get("check_errors_m", [])
    alignment_status = (
        "check_exceeds_match_threshold" if any(error > MAX_DISTANCE_M for error in check_errors)
        else "checked" if check_errors else "provisional"
    )
    route_summary = summarize_routes(assets, rows, alignment_status == "checked")
    report = {
        "alignment_status": alignment_status,
        "prediction_scene": str(PREDICTED_SCENE), "prediction_glb": str(prediction_glb) if prediction_glb else None,
        "blue_reference_json": blue_metadata, "reference_files": assets, "alignment": alignment,
        "max_distance_m": MAX_DISTANCE_M, "unmapped_predictions": unmapped,
        "unassigned_mapped_predictions": len(aligned_positions) - sum(row["prediction_id"] is not None for row in rows),
        "route_summary": route_summary, "holds": rows,
    }
    display(pd.DataFrame(route_summary))
    detail = pd.json_normalize(rows)
    columns = ["route", "annotated_hold_id", "status", "prediction_id", "distance_m"]
    columns += [column for column in detail if column.startswith("reference.")]
    display(detail[columns])
    print("Alignment status", alignment_status, "| unmapped predictions", unmapped)
else:
    print("Evaluation needs a saved 3D scene, an alignment and at least one marked route.")

In [ ]:
if report is not None:
    figure = go.Figure(annotation_wall) if annotation_wall is not None else go.Figure()
    if len(aligned_positions):
        figure.add_scatter3d(
            x=aligned_positions[:, 0], y=aligned_positions[:, 1], z=aligned_positions[:, 2],
            mode="markers", name="Predictions", marker={"size": 4, "color": "#22d3ee"},
        )
    colors = {"matched": "#22c55e", "unmatched": "#fb923c", "tentative_match": "#a78bfa",
              "visibility_unknown": "#64748b", "out_of_scope": "#94a3b8"}
    for status, color in colors.items():
        subset = [row for row in rows if row["status"] == status]
        if subset:
            x, y, z = np.asarray([row["target_position_m"] for row in subset]).T
            figure.add_scatter3d(
                x=x, y=y, z=z, mode="markers", name=status.replace("_", " ").capitalize(),
                marker={"size": 6, "color": color},
                text=[row["annotated_hold_id"] for row in subset], hoverinfo="text",
            )
    lines = [point for row in rows if row["prediction_id"] is not None
             for point in (row["target_position_m"], row["prediction_position_m"], [None, None, None])]
    if lines:
        x, y, z = zip(*lines)
        figure.add_scatter3d(x=x, y=y, z=z, mode="lines", name="Match distance",
                             line={"width": 4, "color": "#f8fafc"}, hoverinfo="skip")
    configure_plot(figure, "Wall C hold comparison", front=-2.5)
    figure.show()

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    report_path = OUTPUT_DIR / "wallc_evaluation.json"
    report_path.write_text(json.dumps(report, indent=2, allow_nan=False) + "\n")
    pd.json_normalize(rows).to_csv(OUTPUT_DIR / "wallc_hold_matches.csv", index=False)
    (OUTPUT_DIR / "glb_to_annotation.json").write_text(json.dumps(alignment, indent=2) + "\n")
    print(report_path)